# Linear regression tutorial

This notebook fits `mlpackage.supervised_learning.LinearRegression`—**ordinary least squares** via the normal equations (`numpy.linalg.pinv`)—on the sklearn **Diabetes** dataset. We split train/test, evaluate **`R_squared`** and **`rmse`**, inspect coefficients, visualize predictions on held-out data, and contrast the fitted model with a trivial **mean baseline**.

## Prerequisites and goals

**You will**
- Load continuous targets with numeric features.
- Split into training vs held-out test rows.
- Solve OLS for intercept + slopes and interpret **`intercept`** / **`coef_`**.
- Measure **`R_squared`** and **`rmse`** on train and test (`LinearRegression` exposes these explicitly—there is no `score` method).
- Plot truth vs predicted on the test fold and compare against a constant predictor.

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `mlpackage`.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

from mlpackage.supervised_learning import LinearRegression

## Step 1: Load Diabetes

Ten baseline variables (plus sex and derived quantities) predict **quantitative disease progression** one year later. We align with the decision-tree regressor tutorial so results stay comparable across models.

In [ ]:
diabetes = load_diabetes()
X = np.asarray(diabetes.data, dtype=float)
y = np.asarray(diabetes.target, dtype=float)

feature_names = list(diabetes.feature_names)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_names)
print("Target stats — mean:", round(float(y.mean()), 2), "std:", round(float(y.std(ddof=0)), 2))

## Step 2: Train/test split

We evaluate **generalization** on rows withheld from `fit`. **`random_state`** fixes reproducibility.

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
)

print("Train:", X_train.shape[0], "  Test:", X_test.shape[0])

## Step 3: Fit OLS `LinearRegression`

Internally the learner stacks a column of ones and solves \(\boldsymbol{\theta} = (\mathbf{X}^\top \mathbf{X})^{+} \mathbf{X}^\top \mathbf{y}\). **No feature scaling is required** for correctness of least squares; scaling only affects coefficient magnitudes relative to one another.

In [ ]:
reg = LinearRegression()
reg.fit(X_train, y_train)

print("Fitted:", reg.fitted)
print("Intercept:", round(reg.intercept, 4))
display(pd.Series(reg.coef_, index=feature_names, name="coef_").round(4))

## Step 4: Predict on train and test

Affine rule: **`y_hat = X @ coef_ + intercept`**.

In [ ]:
y_hat_train = reg.predict(X_train)
y_hat_test = reg.predict(X_test)

print("First 8 test predictions:", np.round(y_hat_test[:8], 2))
print("First 8 test targets:   ", np.round(y_test[:8], 2))

## Step 5: `R_squared` and `rmse`

Use the **same** metric definitions on train and test by passing the matching `(X, y)` pairs.

In [ ]:
print("Train R^2 :", round(reg.R_squared(X_train, y_train), 4))
print("Test R^2  :", round(reg.R_squared(X_test, y_test), 4))
print("Train RMSE:", round(reg.rmse(X_train, y_train), 4))
print("Test RMSE :", round(reg.rmse(X_test, y_test), 4))

## Step 6: Residual inspection on the test fold

Residuals \(e_i = y_i - \hat{y}_i\) reveal systematic bias (patterns vs fitted values are worth plotting for serious analyses).

In [ ]:
residual = y_test - y_hat_test

preview = pd.DataFrame(
    {
        "y_true": y_test,
        "y_pred": y_hat_test,
        "residual": residual,
    }
).head(12)
display(preview.round(3))

## Step 7: Truth vs predicted on held-out data

Points near the diagonal indicate accurate affine predictions on unseen patients.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_hat_test, alpha=0.75, edgecolors="black", linewidths=0.3)

lims = float(min(y_test.min(), y_hat_test.min())), float(max(y_test.max(), y_hat_test.max()))
margin = (lims[1] - lims[0]) * 0.05 if lims[1] > lims[0] else 1.0
lo, hi = lims[0] - margin, lims[1] + margin
plt.plot([lo, hi], [lo, hi], linestyle="--", color="gray", linewidth=1, label="y = ŷ")

plt.xlabel("Actual progression (held-out)")
plt.ylabel("Predicted progression (held-out)")
plt.title("Diabetes — linear model: test truth vs prediction")
plt.legend()
plt.axis("square")
plt.tight_layout()

from pathlib import Path

_nb_here = Path("linear_regression_tutorial.ipynb")
if _nb_here.is_file():
    _plot_path = _nb_here.with_name("diabetes_linear_y_vs_yhat_scatter.png")
else:
    _plot_path = Path(
        "examples/supervised_learning/linear_regression/diabetes_linear_y_vs_yhat_scatter.png"
    )
    _plot_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_plot_path.resolve()}")
plt.show()

## Step 8: Compare against a constant baseline

A straw-man regressor predicts **`mean(y_train)`** for every test sample—no features. If our linear model cannot beat this baseline on test RMSE / \(R^2\), the learned slopes may not help under this split.

In [ ]:
train_mean = float(y_train.mean())
baseline_test = np.full_like(y_test, train_mean, dtype=float)


def r2_manual(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    if ss_tot == 0.0:
        return 1.0
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    return 1.0 - ss_res / ss_tot


def rmse_manual(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


rows = [
    {
        "model": "OLS LinearRegression",
        "test_R2": reg.R_squared(X_test, y_test),
        "test_RMSE": reg.rmse(X_test, y_test),
    },
    {
        "model": f"Baseline (always {train_mean:.1f})",
        "test_R2": r2_manual(y_test, baseline_test),
        "test_RMSE": rmse_manual(y_test, baseline_test),
    },
]

display(pd.DataFrame(rows).round(4))